In [1]:
import psycopg2
from psycopg2 import sql
from psycopg2.extras import execute_values, Json
import logging
import json
import html
import re

In [2]:
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

In [3]:
DB_PARAMS = {
    'dbname':   'thecall',
    'user':     'postgres',
    'password': 'password',
    'host':     'localhost',
    'port':     '5432'
}

TABLE_SCHEMA = 'articles'
TABLE_NAME   = 'filelist'

# Rows per commit. Reduce if you hit memory limits.
BATCH_SIZE = 500

# ── Entity cleaning ───────────────────────────────────────────────────────────
# Updated Map of QXP entity → clean Unicode replacement
# The (?:amp;)? part allows the regex to match both raw '&' and escaped '&amp;'
_QXP_ENTITIES = [
    # Horizontal tab / indent
    (re.compile(r'&(?:amp;)?hTab;'),       '\u2003'),    # em space
    # Punctuation space
    (re.compile(r'&(?:amp;)?punctSpace;'), '\u2008'),    # punctuation space
    # Non-breaking hyphen
    (re.compile(r'&(?:amp;)?nbHyphen;'),   '\u2011'),    # non-breaking hyphen
    # Non-breaking space
    (re.compile(r'&(?:amp;)?nbSpace;'),    '\u00a0'),    # standard &nbsp;
    # Hair space
    (re.compile(r'&(?:amp;)?hairsp;'),     '\u200a'),    # hair space
    # Thin space
    (re.compile(r'&(?:amp;)?thinsp;'),     '\u2009'),    # thin space
    # Figure space
    (re.compile(r'&(?:amp;)?numsp;'),      '\u2007'),    # figure space
    # Discretionary hyphen
    (re.compile(r'&(?:amp;)?shy;'),        '\u00ad'),    # soft hyphen
    # Any remaining unknown &xxx; or &amp;xxx; QXP entities
    (re.compile(r'&(amp;)? [a-zA-Z][a-zA-Z0-9]{2,10};'), ''),
]



In [4]:
# ── Database helpers ──────────────────────────────────────────────────────────

def connect_to_db(params):
    try:
        logging.info("Connecting to PostgreSQL...")
        conn = psycopg2.connect(**params)
        conn.autocommit = False
        logging.info("Connected.")
        return conn
    except psycopg2.Error as e:
        logging.error(f"Connection failed: {e}")
        return None




In [5]:
def fetch_batch(conn, schema, table, batch_size):
    """Fetch rows where json_raw is populated but json_unified not yet written."""
    with conn.cursor() as cur:
        cur.execute(
            sql.SQL("""
                SELECT id, json_raw
                FROM {}.{}
                WHERE json_raw IS NOT NULL
                  AND json_unified IS NULL
                LIMIT %s
            """).format(sql.Identifier(schema), sql.Identifier(table)),
            (batch_size,)
        )
        return cur.fetchall()   # list of (id, json_raw)


In [6]:
def update_batch(conn, schema, table, rows):
    if not rows:
        return

    # Keep this as is; Json(dict) handles the serialization
    values = [(Json(r[0]), r[1]) for r in rows]

    with conn.cursor() as cur:
        execute_values(
            cur,
            sql.SQL("""
                UPDATE {}.{} AS t
                SET json_unified = data.v::jsonb
                FROM (VALUES %s) AS data(v, id)
                WHERE t.id = data.id::integer
            """).format(
                sql.Identifier(schema),
                sql.Identifier(table)
            ).as_string(conn),
            values
        )
    conn.commit()
    logging.info(f"  Committed {len(rows)} rows.")

In [7]:
# ── Format detection ──────────────────────────────────────────────────────────

def detect_format(raw: dict) -> str:
    """
    Returns 'quarkxpress' or 'indesign'.
    QXP json has a 'boxes' key; InDesign has 'pages'.
    """
    if 'boxes' in raw:
        return 'quarkxpress'
    if 'pages' in raw:
        return 'indesign'
    return 'unknown'

In [8]:
# ── Normalisation: QuarkXPress ────────────────────────────────────────────────

def normalise_quark(raw: dict, db_id: int) -> dict:
    frames = []
    for box in raw.get('boxes', []):
        paras = box.get('paragraphs', [])
        frames.append({
            'frame_id':   box.get('box_id'),
            'bounds':     None,
            'paragraphs': [{'style': None, 'text': p} for p in paras],
            'text':       ' '.join(paras),
        })
    return {
        'db_id':         db_id,
        'serial':        raw.get('serial'),
        'source_file':   raw.get('source_file'),
        'source_format': 'quarkxpress',
        'layout_name':   raw.get('layout_name'),
        'frames':        frames,
    }


In [9]:
# ── Normalisation: InDesign ───────────────────────────────────────────────────

def normalise_indesign(raw: dict, db_id: int) -> dict:
    frames = []
    for page in raw.get('pages', []):
        for frame in page.get('frames', []):
            paras = [
                {
                    'style': p.get('style'),
                    'text':  p.get('text', ''),
                }
                for p in frame.get('paragraphs', [])
                if p.get('text', '').strip()
            ]
            frames.append({
                'frame_id':   None,
                'bounds':     frame.get('bounds'),
                'paragraphs': paras,
                'text':       ' '.join(p['text'] for p in paras),
            })
    return {
        'db_id':         db_id,
        'serial':        None,
        'source_file':   raw.get('source_file', raw.get('document_name')),
        'source_format': 'indesign',
        'layout_name':   raw.get('document_name'),
        'frames':        frames,
    }

In [10]:


def clean_qxp_entities(text: str) -> str:
    """Replace QuarkXPress proprietary named entities with Unicode equivalents."""
    for pattern, replacement in _QXP_ENTITIES:
        text = pattern.sub(replacement, text)
    return text


# ── Text rendering ────────────────────────────────────────────────────────────

def make_text_plain(unified: dict) -> str:
    """
    Newline-separated plain text of all paragraphs across all frames.
    Suitable for PostgreSQL full-text search (tsvector) or external indexing.
    """
    lines = []
    if unified.get('layout_name'):
        lines.append(unified['layout_name'])
    for frame in unified.get('frames', []):
        for para in frame.get('paragraphs', []):
            t = para['text'].strip()
            if t:
                lines.append(t)
    return '\n'.join(lines)


def make_text_html(unified: dict) -> str:
    """
    Self-contained HTML fragment ready for direct insertion into a web page.

    Pipeline per paragraph:
      1. html.escape()        — convert < > & " to safe HTML entities
      2. clean_qxp_entities() — replace QXP proprietary entities with Unicode

    Step 1 must come first so that legitimate & characters introduced by
    html.escape are not then consumed by the entity cleaner.

    Structure:
        <article class="ad-card" data-format="quarkxpress" data-id="123">
          <h2 class="ad-title">Layout name</h2>
          <div class="ad-frame">
            <p class="ad-para" data-style="pr-Normal">paragraph text</p>
          </div>
        </article>
    """
    fmt    = unified.get('source_format', 'unknown')
    db_id  = unified.get('db_id', '')
    serial = unified.get('serial') or ''
    title  = clean_qxp_entities(html.escape(unified.get('layout_name') or ''))

    parts = []
    parts.append(
        f'<article class="ad-card" '
        f'data-format="{fmt}" '
        f'data-id="{db_id}" '
        f'data-serial="{serial}">'
    )
    if title:
        parts.append(f'  <h2 class="ad-title">{title}</h2>')

    for frame in unified.get('frames', []):
        frame_id = html.escape(str(frame.get('frame_id') or ''))
        parts.append(f'  <div class="ad-frame" data-frame-id="{frame_id}">')
        for para in frame.get('paragraphs', []):
            # Step 1: escape HTML special chars
            text  = html.escape(para['text'].strip())
            # Step 2: clean QXP entities that survived into the text
            text  = clean_qxp_entities(text)
            style = html.escape(str(para.get('style') or ''))
            if text:
                parts.append(
                    f'    <p class="ad-para" data-style="{style}">{text}</p>'
                )
        parts.append('  </div>')

    parts.append('</article>')
    return '\n'.join(parts)

In [11]:
# ── Core transform ────────────────────────────────────────────────────────────

def transform(db_id: int, json_raw) -> dict | None:
    """
    Accepts the raw value from the DB.
    psycopg2 auto-parses jsonb columns to dicts; text/json columns arrive
    as strings and are parsed here. Returns the unified dict, or None on error.
    """
    try:
        raw = json_raw if isinstance(json_raw, dict) else json.loads(json_raw)
    except (json.JSONDecodeError, TypeError) as e:
        logging.warning(f"  id={db_id}: cannot parse json_raw — {e}")
        return None

    fmt = detect_format(raw)
    if fmt == 'quarkxpress':
        unified = normalise_quark(raw, db_id)
    elif fmt == 'indesign':
        unified = normalise_indesign(raw, db_id)
    else:
        logging.warning(f"  id={db_id}: unknown format, skipping.")
        return None

    unified['text_plain'] = make_text_plain(unified)
    unified['text_html']  = make_text_html(unified)
    return unified

In [12]:
# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    conn = connect_to_db(DB_PARAMS)
    if not conn:
        return

    total_processed = 0
    total_errors    = 0

    try:
        while True:
            rows = fetch_batch(conn, TABLE_SCHEMA, TABLE_NAME, BATCH_SIZE)
            if not rows:
                break

            logging.info(f"Processing batch of {len(rows)} rows...")
            updates = []

            for db_id, json_raw in rows:
                unified = transform(db_id, json_raw)
                if unified is None:
                    total_errors += 1
                    continue
                updates.append((unified, db_id))

            update_batch(conn, TABLE_SCHEMA, TABLE_NAME, updates)
            total_processed += len(updates)

    finally:
        conn.close()
        logging.info("Database connection closed.")
        logging.info(f"Done. Processed: {total_processed}  Errors: {total_errors}")


if __name__ == '__main__':
    main()

INFO: Connecting to PostgreSQL...
INFO: Connected.
INFO: Processing batch of 14 rows...
INFO:   Committed 14 rows.
INFO: Database connection closed.
INFO: Done. Processed: 14  Errors: 0


In [10]:
# ── ONE-TIME: Reset json_unified so the corrected script can reprocess all rows ──
# Run this cell ONCE before running main().
# It sets json_unified back to NULL for every row so fetch_batch picks them up again.
# Comment it out after the first run.

# conn = connect_to_db(DB_PARAMS)
# if conn:
#     with conn.cursor() as cur:
#         cur.execute(
#             sql.SQL("UPDATE {}.{} SET json_unified = NULL WHERE json_raw IS NOT NULL").format(
#                 sql.Identifier(TABLE_SCHEMA), sql.Identifier(TABLE_NAME)
#             )
#         )
#         logging.info(f"Reset {cur.rowcount:,} rows to json_unified = NULL.")
#     conn.commit()
#     conn.close()

INFO: Connecting to PostgreSQL...
INFO: Connected.
INFO: Reset 141,125 rows to json_unified = NULL.


In [ ]:
# ── Spot-check a single row without writing to the DB ────────────────────────

CHECK_ID = 415579  # <-- change this to any id in your table

conn = connect_to_db(DB_PARAMS)
if conn:
    with conn.cursor() as cur:
        cur.execute(
            sql.SQL("SELECT id, json_raw FROM {}.{} WHERE id = %s").format(
                sql.Identifier(TABLE_SCHEMA), sql.Identifier(TABLE_NAME)
            ),
            (CHECK_ID,)   # parameter tuple — fixes the missing-arg bug in v2
        )
        row = cur.fetchone()
    conn.close()

    if row:
        result = transform(row[0], row[1])
        if result:
            print("=== UNIFIED (excluding text fields) ===")
            preview = {k: v for k, v in result.items() if k not in ('text_plain', 'text_html')}
            print(json.dumps(preview, indent=2, ensure_ascii=False))

            print("\n=== TEXT_PLAIN ===")
            print(result['text_plain'])

            print("\n=== TEXT_HTML ===")
            print(result['text_html'])
    else:
        print(f"No row found with id={CHECK_ID}")